In [0]:
df = spark.read.csv('/Volumes/workspace/default/emp1/emp_1.csv', header= True)
df.show()

In [0]:
df1 = spark.read.csv('/Volumes/workspace/default/emp1/emp_1.csv', header=True)
df1.show()

In [0]:
df1.printSchema()

In [0]:
# For proper schema we have to use inferSchema
df2 = spark.read.csv('/Volumes/workspace/default/emp1/emp_1.csv', header=True, inferSchema=True)
df2.printSchema()

In [0]:
#“This code is reading an existing CSV file from a volume path and converting it into a DataFrame. It does not create a new file.”
df3 = spark.read.format("csv").options(header=True, inferScehma=True).load('/Volumes/workspace/default/emp1/emp_1-.csv')

In [0]:
#To select any particular col
df2.select("emp_id","email").show()

In [0]:
df2.select(df2.columns[1:5]).show()

In [0]:
df.withColumn("salaryAfterIncrement", df.salary * 10).show()

In [0]:
#For this we have to use lit function
from pyspark.sql.functions import lit
df.withColumn("country",lit("India")).show()

In [0]:
from pyspark.sql.functions import col
df4 = df.withColumn("salary",col("salary").cast("Integer"))
df4.printSchema()

In [0]:
df.withColumnRenamed("loc","location").show()

# Filter in Pyspark

In [0]:
df.filter(df.address == "india").show()
df.filter((df.name == 'manish') & (df.salary == 10000)).show() #for OR condition use |

In [0]:
#Fetching the record where name start with r
df.filter(df.name.startswith("r")).show() 

#Fetching the record where name end with l
df.filter(df.name.endswith("l")).show()

#Fetching the record where name is having ul
df.filter(df.name.like("%ul%")).show()

# Distinct() and dropDuplicates()

In [0]:
df1 = df.distinct() # will get the unique record
df1.show()

In [0]:
df2 = df.dropDuplicates(['emp_id', 'salary'])
df2.show()

# Sort() and orderBy()

In [0]:
df.sort("salary").show() #by default ascending
df.sort(df.salary.desc()).show()
df.orderBy(df.salary.desc()).show()

# GroupBy

In [0]:
df.groupBy("address").sum("salary").show()
df.groupBy("address","emp_id").sum("salary").show()
df.groupBy("address").max("salary").show()

# JOIN

In [0]:
emp_df.join(dept_df, emp_df.emp_id == dept_df.user, 'inner').show()
emp_df.join(dpt_df, emp_df.emp_id == dept_df.user, 'left').show()

# Union & UnionAll


In [0]:
import pyspark 
from pyspark.sql import SparkSession
data1 = [
    ("Aman", "IT", "Delhi", 50000, 25, 5000),
    ("Riya", "HR", "Mumbai", 45000, 28, 4000),
    ("Karan", "Finance", "Pune", 60000, 30, 7000),
    ("Sneha", "IT", "Bangalore", 55000, 27, 6000),
    ("Rahul", "Sales", "Chennai", 40000, 26, 3000)
]
columns = ["elmloyee_name","department", "state", "salary", "age", "bonus"]
df = spark.createDataFrame(data1,schema = columns)
df.show()

In [0]:
import pyspark 
from pyspark.sql import SparkSession
data2 = [
     ("Aman", "IT", "Delhi", 50000, 25, 5000),
    ("Priya", "Marketing", "Hyderabad", 48000, 29, 4500),
    ("Arjun", "IT", "Noida", 62000, 31, 8000),
    ("Sneha", "IT", "Bangalore", 55000, 27, 6000),
    ("Neha", "Finance", "Kolkata", 53000, 26, 5000)
]
columns2 = ["elmloyee_name","department", "state", "salary", "age", "bonus"]
df2 = spark.createDataFrame(data2,schema = columns2)
df2.show()

In [0]:
df.union(df2).show()

# FILL and FILLNA

In [0]:
df = spark.read.csv('/Volumes/workspace/default/samledata/sample.csv', header=True)
df.show()

In [0]:
#filling null values with blank value
df.na.fill("").show()

#filling null values with unknown value
df.na.fill("unknown").show() #for this col having integer value or any value other than string will not be changed

#replacing any specific col
df.na.fill("",["city"]).show()

#filling city col with blank value and population with unknown val
df.na.fill("",["city"]).na.fill("unknown",["population"]).show()

In [0]:
df.collect()

# StructType and StructField IMP

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType
# for this data we are not defining any col names
data = [
    (1, 'manish', 'usa'),
    (2, 'mani', 'usa'),
    (1, 'nish', 'usa')
]

schema = StringType([StructField(name="id", dataType=IntegerType()),
                     StructField(name="name", dataType=StringType()),
                     StructField(name="location", dataType=StringType())])

df= spark.createDataFrame(data, schema)
df.show()
df.printScehma()

# Pivot and UnPivot

In [0]:
data = [('Banana',1000, "USA"),("Carrot", 2000, "india"),("Organge", 200, "Nepal"),('Banana',100, "India")]
col = ["Product", "Amount", "Country"]
df = spark.createDataFrame(data, col)
df.show()

In [0]:
df.groupBy("Product").pivot("Country").sum("Amount").show()

# UDF Functions

In [0]:
#creating a new datagrame
data = [("Finance",10), ("Marketing",20), ("Sales",30),("IT", 40)]
col = ["dept_name", "dept_id"]
df = spark.createDataFrame(data,col)
df.show()

In [0]:
from pyspark.sql.types import LongType
def addone(a):
    return a+1
addone_df = udf(addone, LongType())

#using this UDF fucntion 
df.select("dept_name", "dept_id", addone_df("dept_id")).show()

# Transform fun

In [0]:
data = [ ([1, 2, 3],),  ([4, 5, 6],)]
df = spark.createDataFrame(data, ["numbers"])
df_new = df.withColumn(
    "updated_numbers",
    transform(col("numbers"), lambda x: x + 1))
df_new.show(truncate=False)


# Creating temp View in Pyspark

In [0]:
#creating df
import pyspark 
from pyspark.sql import SparkSession
data2 = [
     ("Aman", "IT",5000),
    ("Priya", "Marketing",4500),
    ("Arjun", "IT",8000),
    ("Sneha", "IT",6000),
    ("Neha", "Finance",5000)
]
columns2 = ["elmloyee_name","department","salary"]
df = spark.createDataFrame(data2, columns2)

df.createOrReplaceTempView("employee")

In [0]:
%sql
select * from employee;

# Window fucntion in PySpark

# Interview Que

In [0]:
# 1.Print duplicate records present in the data(take ex. We have id and duplicated emails)
# 2. Remove duplicate records

#creating DF
data = [(1,'abc@gmail.com'), (2,'def@gmail.com'), (3,'abc@gmail.com')]
col = ['id','email']
df = spark.createDataFrame(data,col)
df.show()

# printing duplicate record
from pyspark.sql.functions import col
display(df.groupBy('email').count().filter(col('count')>1))

#remove duplicate record
df.distinct().show() 
# OR
df.dropDuplicates().show()
#for removing duplicate for specific col
df.distinct(['email']).show() # ye galat hai distinct mai col pass nhi kr skte hain
df.dropDuplicates(['email']).show() #this will drop the duplicate email records

In [0]:
# We are having two tables CUSTOMER and ORDER Table 
# We have to find out customer who have not order anything
# We have to find out customers who have ordred

data = [(1,'Manish'),(2,'Rahul'),(3,'Monu'),(4,'Ram')]
col= ['cust_id','cust_name']
df_customer = spark.createDataFrame(data,col)
df_customer.show()

data= [(1,4),(3,2)]
col = ['order_id','cust_id']
df_order = spark.createDataFrame(data,col)
df_order.show()

#finding customers who have not ordred anything
display(df_customer.join(df_order, df_customer.cust_id == df_order.cust_id,"left").filter(df_order.cust_id.isNull()))

#finding customers who have ordered
display(df_customer.join(df_order, df_customer.cust_id == df_order.cust_id,"inner"))

In [0]:
# We have 2 tables EMPLOYEE and DEPARTMENT Table is given
# 1.Find the highest salary based on each department
# 2.Find the employee who is getting the highest salary based on each department
# 3.Find the lowest salary based on each department name
# 4. Find the employee who is getting the lowest salary based on each dept

#creating DF
data = [('Manish',1,75000),('Raghav',1,8000),('surya',2,70000),('virat',2,75000),('rohit',2,8500),('ram',3,8000),('priya',3,9000),('shyam',4,95000),('sachin',4,10000)]
col = ['emp_name','dept_id','salary']
df_employee = spark.createDataFrame(data,col)
df_employee.show()


data1 = [(1,'DATA_ENGINEER'),(2,'SALES'),(3,'SOFTWARE'),(4,'HR')]
col1 = ['dept_id','dept_name']
df_department = spark.createDataFrame(data1,col1)
df_department.show()

#1
from pyspark.sql.functions import *
df = df_employee.join(df_department, df_employee.dept_id == df_department.dept_id,'left')
df.groupBy("dept_name").agg(max("salary").alias("max_salary")).show()


In [0]:
#2
from pyspark.sql.functions import *
df = df_employee.join(df_department, df_employee.dept_id == df_department.dept_id,'left')
df1 = df.groupBy("dept_name").agg(max("salary").alias("max_salary"))
# we will first find out the max salary dept wise then will do join on df where we have done join to find employee name
df2 = df.join(df1,df.dept_name == df1.dept_name, "inner")
df2.filter(col('salary') == col('max_salary')).show()

In [0]:
#3
from pyspark.sql.functions import *
df = df_employee.join(df_department, df_employee.dept_id == df_department.dept_id, "left")
df.groupBy("dept_name").agg(min("salary").alias("min_salary")).show()

In [0]:
#4
from pyspark.sql.functions import *
df = df_employee.join(df_department, df_employee.dept_id == df_department.dept_id, "left")
df1 = df.groupBy("dept_name").agg(min("salary").alias("min_salary"))
df2 = df.join(df1, df.dept_name == df1.dept_name, "inner")
df2.filter(col("salary") == col("min_salary")).show()

In [0]:
# 1. We have to flatten data from 1 row to multiple row eg. [1,2,3] we need to convert into row wise flattening
# 2.We have to find out first not null value
#1
from pyspark.sql.functions import *
data = [(1,['mobile','PC','Table']),(2,['mobile','PC']),(3,['Tab','Pen'])]
schema = ['cust_id','product_purchase']

df = spark.createDataFrame(data,schema)
df.withColumn('product',explode('product_purchase')).select('cust_id','product').show()

In [0]:
#2
data = [(1,'yes',None,None),(2,None,'yes',None),(3,None,'no','yes')]
schema = ['cust_id','device_using1','device_using2','device_using3']
df = spark.createDataFrame(data,schema)
#for finding 1st not null value we should use coalesce function
df.withColumn('new',coalesce(col('device_using1'),col('device_using2'),col("device_using3"))).show()


In [0]:
# Que. 1.We are getting data in the form of string JSON we need to convert into JSON format
# 2.After converting we need to create separate column from json body.
data = [('Manish','{"street":"123 St", "city":"Delhi"}'),('Ram','{"street":"465 St", "city":"Mumbai"}')]
schema = ['name', 'address']
df = spark.createDataFrame(data,schema)
display(df)

#creating table so that we can write SQL query to solve this
df.createOrReplaceTempView('sample')

In [0]:
%sql
with mycte as(
select name,address,from_json(address, 'street string, city string') as address_new from sample)
-- 2
select name, address, address_new.street as street, address_new.city as city from mycte;


In [0]:
from pyspark.sql.functions import *
#1
df1 = df.withColumn('address_new', from_json(col('address'),'street string, city string'))
display(df1)

#2
df1.select('name','address','address_new',col('address_new').street.alias('street'),col('address_new').city.alias('city')).display()

In [0]:
# Que. 1.Find out cummulative sales or running total sales
# 2.find out the prev sales
# 3.find out the next sales

#creating DF
data = [['2024-01-01',20000],['2024-01-02',10000], ['2024-01-03',10000], ['2024-01-04',10000]]
col = ['date','sales']
df =spark.createDataFrame(data,col)
df.show()

from pyspark.sql import Window
from pyspark.sql.functions import *

#1
window = Window.orderBy('date')
df.withColumn("cummulative_sales",sum(col('sales')).over(window)).show()

#2
window1 = Window.orderBy('date')
df.withColumn("prev_sales",lag(col('sales')).over(window1)).show()

#3
window2 = Window.orderBy('date')
df.withColumn("next_sales",lead(col('sales')).over(window2)).show()

In [0]:
# Que.Find out the missing number
data = [(1,),(2,),(5,),(7,),(8,),(10,)]
column = ['ID']
df = spark.createDataFrame(data,column)
df.show()

#for this we will create a DF having all no. from 1 to 10 then will subract this DF to old DF to get missing no.
from pyspark.sql.types import IntegerType
list_new = range(1,11,1)
df_new = spark.createDataFrame(list_new,IntegerType())
df_new.show()

#getting missing numbers
display(df_new.subtract(df))

In [0]:
# Que. Group multiple rows into single
data = [(1,'Manish','Mobile'),(1,'Manish','Washing Machine'),(2,'Rahul','Car'),(2,'Rahul','Mobile'),(2,'Rahul','Scooty'),(3,'Monu','Scooty')]
schema = ['Customer_ID','Customer_Name','Purchase']
df = spark.createDataFrame(data,schema)
df.show()

from pyspark.sql.functions import *
df.groupBy('Customer_ID','Customer_Name').agg(collect_set('purchase')).show()

In [0]:
# Que. How to combine many list
list1 = ["a","b","c","d"]
list2 = [1,2,3,4]
rdd = spark.sparkContext.parallelize(list(zip(list1,list2)))
print(rdd)
df = rdd.toDF(["Column1","Column2"])
df.show()